# V2 Spec & Registry: Declarative Configuration

BaseAttentive v2 introduces a **registry / resolver / assembly** system
that decouples model *description* from model *construction*.  This
notebook walks you through:

1. `BaseAttentiveSpec` — a frozen, backend-neutral model spec
2. `BaseAttentiveComponentSpec` — per-component configuration
3. JSON serialisation and reload
4. `ComponentRegistry` — registering a custom encoder
5. `ModelRegistry` — building a model from a spec
6. `BaseAttentiveV2Assembly` — full resolver/assembler pipeline

## Why declarative configuration?

The keyword-argument style (`BaseAttentive(embed_dim=64, ...)`) is
convenient but couples the model to its Python constructor.  Specs let
you:

- **Version-control** model configs as plain JSON
- **Reproduce** any experiment exactly — share the JSON, rebuild the model
- **Swap backends** (`tensorflow` / `torch` / `jax`) without touching model code
- **Register custom components** and reference them by string key

## Setup

In [ ]:
import json
import numpy as np

from base_attentive import BaseAttentive
from base_attentive.config import (
    BaseAttentiveSpec,
    BaseAttentiveComponentSpec,
)
from base_attentive.registry import (
    ComponentRegistry,
    ModelRegistry,
)
from base_attentive.assembly import BaseAttentiveV2Assembly

print("V2 spec imports OK")

---

## 1 — Keyword approach (recap)

The classic keyword approach is fine for quick experiments.  Everything
shown here also applies to v1-style construction.

In [ ]:
model_kw = BaseAttentive(
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    embed_dim=64,
    num_heads=8,
    dropout_rate=0.15,
)
print("Keyword model:", model_kw.name)

---

## 2 — `BaseAttentiveSpec`

`BaseAttentiveSpec` is a **frozen dataclass** that holds all
configuration for a BaseAttentive model.  It is backend-neutral — the
same spec object works with TensorFlow, PyTorch, or JAX.

### Field groups

| Group | Fields |
|-------|-------|
| **Required I/O** | `static_input_dim`, `dynamic_input_dim`, `future_input_dim`, `output_dim`, `forecast_horizon` |
| **Capacity** | `embed_dim`, `attention_units`, `num_heads`, `ff_units`, `dropout_rate` |
| **Encoder** | `objective`, `num_encoder_layers`, `scales`, `multi_scale_agg` |
| **Attention / Decoder** | `attention_levels`, `num_decoder_layers`, `use_residuals` |
| **Output** | `output_mode`, `quantiles`, `n_mixture_components` |
| **Components** | `component_specs` — list of `BaseAttentiveComponentSpec` |
| **Backend** | `backend` — `"tensorflow"`, `"torch"`, or `"jax"` |

In [ ]:
spec = BaseAttentiveSpec(
    # I/O
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    # Capacity
    embed_dim=64,
    num_heads=8,
    dropout_rate=0.15,
    # Architecture
    objective="hybrid",
    num_encoder_layers=2,
    attention_levels=["cross", "hierarchical"],
    # Output
    quantiles=[0.1, 0.5, 0.9],
    # Backend
    backend="tensorflow",   # change to "torch" or "jax" as needed
)

print("spec.embed_dim     :", spec.embed_dim)
print("spec.quantiles     :", spec.quantiles)
print("spec.backend       :", spec.backend)

### Specs are immutable

Specs are frozen dataclasses — attempting to mutate one raises a
`FrozenInstanceError`.  To "modify" a spec, use `dataclasses.replace`:

In [ ]:
from dataclasses import replace

spec_torch = replace(spec, backend="torch", dropout_rate=0.2)

print("Original backend:", spec.backend,       "dropout:", spec.dropout_rate)
print("Modified backend:", spec_torch.backend, "dropout:", spec_torch.dropout_rate)

---

## 3 — JSON Serialisation

Specs serialise to/from plain JSON — no pickles, no framework artefacts.

In [ ]:
# Serialise to dict → JSON string
spec_dict = spec.to_dict()         # built-in method on BaseAttentiveSpec
spec_json = json.dumps(spec_dict, indent=2)

print(spec_json[:600], "...")

In [ ]:
# Reload from JSON
reloaded_dict = json.loads(spec_json)
spec_reload   = BaseAttentiveSpec.from_dict(reloaded_dict)

assert spec_reload.embed_dim == spec.embed_dim
assert spec_reload.quantiles == spec.quantiles
print("Spec round-trip: OK")
print("Reloaded backend:", spec_reload.backend)

In [ ]:
# Save to file
import pathlib

spec_path = pathlib.Path("model_spec.json")
spec_path.write_text(spec_json)
print(f"Spec saved to: {spec_path.resolve()}")

# Load from file
spec_from_file = BaseAttentiveSpec.from_dict(
    json.loads(spec_path.read_text())
)
print("Loaded from file — embed_dim:", spec_from_file.embed_dim)

---

## 4 — `BaseAttentiveComponentSpec`

Each component in the model (encoder, attention layer, VSN, etc.) can
be individually configured through a `BaseAttentiveComponentSpec`.  This
is an optional extension: without component specs the assembly uses
built-in defaults.

### Component spec fields

| Field | Type | Purpose |
|-------|------|--------|
| `component_type` | `str` | Registry key, e.g. `"encoder.lstm"` |
| `name` | `str` | Human-readable label |
| `params` | `dict` | Keyword args forwarded to the builder |
| `enabled` | `bool` | Toggle without removing from spec |

In [ ]:
encoder_spec = BaseAttentiveComponentSpec(
    component_type="encoder.lstm",
    name="my_lstm_encoder",
    params={
        "units": 128,
        "num_layers": 2,
        "return_sequences": True,
    },
    enabled=True,
)

attn_spec = BaseAttentiveComponentSpec(
    component_type="attention.hierarchical",
    name="hierarchical_attn",
    params={"num_heads": 8, "key_dim": 32},
    enabled=True,
)

# Attach to the main spec
spec_with_components = replace(
    spec,
    component_specs=[encoder_spec, attn_spec],
)

print("Component specs attached:",
      [cs.component_type for cs in spec_with_components.component_specs])

---

## 5 — `ComponentRegistry`: Registering a Custom Encoder

The `ComponentRegistry` stores a mapping from string keys
(`"<category>.<name>"`) to **builder callables**.  Any Keras layer can
be registered as a custom component.

### Step 1 — Define a Keras layer

In [ ]:
import keras

class ResidualBiLSTMEncoder(keras.Layer):
    """Bidirectional LSTM with residual connections for long sequences."""

    def __init__(self, units: int = 64, num_layers: int = 2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.num_layers = num_layers
        self._lstms = [
            keras.layers.Bidirectional(
                keras.layers.LSTM(units, return_sequences=True)
            )
            for _ in range(num_layers)
        ]
        self._proj = keras.layers.Dense(units)

    def call(self, x, training=False):
        h = x
        for lstm in self._lstms:
            h_new = lstm(h, training=training)
            # Residual: project and add if dims match
            if h_new.shape[-1] == h.shape[-1]:
                h = h + h_new
            else:
                h = self._proj(h_new)
        return h


print("Custom encoder class defined")

### Step 2 — Write a builder function

The registry stores **builder functions**, not class instances.  A
builder receives the spec's `params` dict and returns a layer.

In [ ]:
def build_residual_bilstm(params: dict) -> ResidualBiLSTMEncoder:
    """Builder for ResidualBiLSTMEncoder."""
    return ResidualBiLSTMEncoder(
        units=params.get("units", 64),
        num_layers=params.get("num_layers", 2),
        name=params.get("name", "residual_bilstm"),
    )

print("Builder function defined")

### Step 3 — Register and verify

In [ ]:
REGISTRY_KEY = "encoder.residual_bilstm"

ComponentRegistry.register(
    key=REGISTRY_KEY,
    builder=build_residual_bilstm,
    description="Bidirectional LSTM with residual connections",
)

# Verify registration
print("Registered encoder keys:")
for key in ComponentRegistry.list_keys(category="encoder"):
    print(" ", key)

In [ ]:
# Test the builder directly
layer = ComponentRegistry.resolve(
    key=REGISTRY_KEY,
    params={"units": 128, "num_layers": 3, "name": "test_enc"},
)
print("Resolved layer type:", type(layer).__name__)
print("Layer units:", layer.units)

### Use the custom encoder in a spec

In [ ]:
custom_enc_spec = BaseAttentiveComponentSpec(
    component_type=REGISTRY_KEY,
    name="my_residual_bilstm",
    params={"units": 96, "num_layers": 2},
    enabled=True,
)

spec_custom = replace(
    spec,
    component_specs=[custom_enc_spec],
)

print("Custom encoder component type:",
      spec_custom.component_specs[0].component_type)

---

## 6 — `ModelRegistry` & Building from Spec

`ModelRegistry` wraps `ComponentRegistry` with model-level logic:
it resolves a full `BaseAttentiveSpec` into a trainable Keras model
via `BaseAttentiveV2Assembly`.

In [ ]:
# Register our model blueprint
ModelRegistry.register(
    name="my_hybrid_v2",
    spec=spec,
    description="Hybrid v2 model for multi-output time-series forecasting",
)

print("Registered model blueprints:")
for name in ModelRegistry.list_names():
    print(" ", name)

In [ ]:
# Build from registry name
model_from_registry = ModelRegistry.build(name="my_hybrid_v2")

print("Model type:", type(model_from_registry).__name__)
print("Forecast horizon:", model_from_registry.forecast_horizon)

---

## 7 — Full Assembly Pipeline: `BaseAttentiveV2Assembly`

`BaseAttentiveV2Assembly` is the low-level resolver that reads a spec,
resolves each component from the registries, and wires everything
together into a complete Keras model.

In [ ]:
assembler = BaseAttentiveV2Assembly(spec=spec_custom)

# build() resolves all components and returns the model
model_assembled = assembler.build()

print("Assembled model name  :", model_assembled.name)
print("Embed dim from spec   :", spec_custom.embed_dim)
print("Encoder from registry :",
      spec_custom.component_specs[0].component_type)

In [ ]:
# Quick smoke-test: forward pass
B, T, H = 8, 24, 24
x_s = np.random.randn(B, 4).astype("float32")
x_d = np.random.randn(B, T, 8).astype("float32")
x_f = np.random.randn(B, H, 6).astype("float32")

out = model_assembled([x_s, x_d, x_f])
print("Output shape:", out.shape if not isinstance(out, (list, tuple))
      else [o.shape for o in out])

---

## 8 — Sharing Specs Across Experiments

A common pattern is to define a **base spec** and derive per-experiment
variants from it:

In [ ]:
BASE_SPEC = BaseAttentiveSpec(
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    embed_dim=64,
    num_heads=8,
    dropout_rate=0.1,
)

# Experiment A: TensorFlow, quantile output
spec_A = replace(BASE_SPEC,
    backend="tensorflow",
    quantiles=[0.1, 0.5, 0.9],
    dropout_rate=0.1,
)

# Experiment B: PyTorch, mixture output, bigger capacity
spec_B = replace(BASE_SPEC,
    backend="torch",
    output_mode="mixture",
    n_mixture_components=3,
    embed_dim=128,
    dropout_rate=0.2,
)

# Experiment C: JAX, gaussian output
spec_C = replace(BASE_SPEC,
    backend="jax",
    output_mode="gaussian",
)

# Save all three to JSON
experiment_specs = {"A": spec_A, "B": spec_B, "C": spec_C}
for name, s in experiment_specs.items():
    pathlib.Path(f"spec_exp_{name}.json").write_text(
        json.dumps(s.to_dict(), indent=2)
    )
    print(f"spec_exp_{name}.json saved — backend={s.backend}, "
          f"output_mode={getattr(s, 'output_mode', 'point')}")

---

## Summary

| Concept | Purpose |
|---------|--------|
| `BaseAttentiveSpec` | Frozen, JSON-serialisable model blueprint |
| `BaseAttentiveComponentSpec` | Per-component override (encoder, attention, etc.) |
| `ComponentRegistry.register()` | Plug in a custom Keras layer by string key |
| `ComponentRegistry.resolve()` | Instantiate a component from key + params |
| `ModelRegistry.register()` | Store a named model blueprint |
| `ModelRegistry.build()` | Build a trainable model from a registry name |
| `BaseAttentiveV2Assembly` | Low-level resolver: spec → complete Keras model |

## Next Steps

- [06_crps_probabilistic_forecasting.ipynb](06_crps_probabilistic_forecasting.ipynb) — probabilistic output heads
- [Architecture Guide](https://base-attentive.readthedocs.io/en/latest/architecture_guide.html) — full V2 architecture documentation
- [Usage Guide](https://base-attentive.readthedocs.io/en/latest/usage.html) — all configuration options